# Lezione 14 — `tf.data`: le pipeline di input

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 10-11 - come si costruisce e addestra una rete Keras, e cosa sono i batch (piccoli gruppi di esempi elaborati insieme a ogni passo di training).

**Cosa saprai fare alla fine:** costruire pipeline di input con `tf.data` —
shuffle, batch, map, prefetch — sapendo cosa fa ciascun pezzo e quali
trappole nasconde (la più famosa: il buffer dello shuffle).

## Parte 1 — Teoria: perché esistono le pipeline di input

Finora: `fit(X, y)` con tutto in memoria. Funziona finché i dati stanno in
RAM e non richiedono lavoro per esempio. I sistemi veri hanno dataset da
gigabyte su disco, trasformazioni per esempio (decodificare immagini,
tokenizzare testo) e una GPU che **non deve mai aspettare i dati**.

`tf.data.Dataset` risolve tutto questo con un'idea sola: il dataset è una
**pipeline pigra** — una catena di trasformazioni che produce esempi al
volo, mentre il modello si addestra, invece di materializzare tutto prima.

I quattro operatori del lavoro quotidiano:

- **`shuffle(buffer)`** — mescola gli esempi. Attenzione, è la trappola
  classica: mescola dentro una finestra di `buffer` elementi, non
  globalmente. Con `buffer` piccolo su dati ordinati (es. per data, come i
  nostri!), i batch restano quasi ordinati. Regola: `buffer` ≥ dimensione
  del dataset quando possibile.
- **`batch(32)`** — impacchetta gli esempi in gruppi di 32, gli stessi batch della Lezione 11 (i gruppi su cui si calcola un gradiente alla volta, invece che su tutto il dataset insieme).
- **`map(f)`** — applica una trasformazione a ogni esempio, al volo e in
  parallelo (`num_parallel_calls=tf.data.AUTOTUNE`).
- **`prefetch(AUTOTUNE)`** — prepara i prossimi batch *mentre* il modello
  lavora sul corrente: la sovrapposizione che tiene occupata la GPU.

L'**ordine conta**: `shuffle` → `batch` (mescoli esempi, non batch interi),
e `prefetch` per ultimo.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import keras

keras.utils.set_random_seed(42)
print('Keras', keras.__version__)

In [ ]:
import tensorflow as tf

numeri = tf.data.Dataset.range(10)

piccolo_buffer = list(numeri.shuffle(2, seed=0).as_numpy_iterator())
grande_buffer = list(numeri.shuffle(10, seed=0).as_numpy_iterator())

print('dati ordinati        :', list(range(10)))
print('shuffle(buffer=2)    :', piccolo_buffer, ' <- quasi ancora ordinati!')
print('shuffle(buffer=10)   :', grande_buffer)

Eccola, la trappola: con `buffer=2` l'ordine originale è ancora
riconoscibile. Se i tuoi dati sono ordinati per data (come l'archivio del
progetto!) e il buffer è piccolo, ogni batch contiene memorie dello stesso
periodo — il gradiente vede "epoche di aprile", poi "epoche di maggio": un
bias sottile che nessun errore ti segnala.

## Parte 2 — Esercizio guidato

Costruisci **tu** la pipeline: dai 10 numeri, produci batch da 3 con
shuffle completo, e con un `map` che eleva al quadrato ogni elemento.
Ordine giusto degli operatori: `shuffle` → `map` → `batch`. Poi stampa i
batch di un'epoca e verifica che i quadrati ci siano tutti (la somma totale
deve essere 285).

**Prova tu:**

In [ ]:
# Scrivi qui: numeri -> shuffle(10) -> map(quadrato) -> batch(3)
# poi somma tutti gli elementi prodotti e verifica == 285

pass

### Soluzione spiegata

- `map` con una lambda TF lavora sui tensori, un esempio alla volta;
- lo shuffle cambia l'ordine, non il contenuto: la somma resta 285 —
  l'assert è il contratto che la pipeline non perde ne' duplica esempi;
- in una pipeline reale aggiungeresti `prefetch(tf.data.AUTOTUNE)` in coda.

In [ ]:
pipeline = (tf.data.Dataset.range(10)
            .shuffle(10, seed=1)
            .map(lambda x: x * x)
            .batch(3))

totale = 0
for batch in pipeline:
    print(batch.numpy())
    totale += int(batch.numpy().sum())
assert totale == sum(i * i for i in range(10)) == 285
print('somma verificata:', totale)

## Parte 3 — Il progetto: Memory AI Lab, passo 14

L'input del training del progetto diventa una pipeline `tf.data` fatta
bene: shuffle con buffer pieno (i nostri dati sono ordinati per tempo — la
trappola della Parte 1 ci riguarda direttamente), batch, prefetch. Stessa rete della Lezione 12 (32 neuroni, dropout, early stopping), stessa valutazione: la pipeline non deve cambiare i
risultati, deve cambiare la **forma** dell'input.

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv') for n in ['train', 'val']}
X = {n: f.drop(columns='target').to_numpy().astype('float32') for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}

train_ds = (tf.data.Dataset.from_tensor_slices((X['train'], y['train']))
            .shuffle(len(y['train']), seed=42)      # buffer = TUTTO il dataset
            .batch(32)
            .prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X['val'], y['val'])).batch(32)

modello = keras.Sequential([
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])
modello.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                metrics=['accuracy'])
modello.fit(train_ds, epochs=300, validation_data=val_ds,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                                     restore_best_weights=True)],
            verbose=0)
print(f"validation con pipeline tf.data: {modello.evaluate(val_ds, verbose=0)[1]:.0%}")

Nota cosa è cambiato in `fit`: niente più `X, y` — riceve direttamente la
pipeline, che d'ora in poi potrà leggere da file, trasformare testo al volo
(serve nella prossima lezione!) e scalare a dati che non stanno in memoria,
senza toccare il modello.

## Cosa hai imparato

- `tf.data` = pipeline **pigra**: gli esempi si producono mentre il modello
  lavora; `prefetch` sovrappone dati e calcolo.
- **`shuffle(buffer)` mescola solo dentro il buffer**: con dati ordinati e
  buffer piccolo i batch restano ordinati — la trappola silenziosa.
- Ordine canonico: `shuffle` → `map` → `batch` → `prefetch`.
- La pipeline cambia la forma dell'input, non i risultati: si verifica con
  un contratto (stessa somma, stessa accuratezza attesa).

## Quiz

1. Dataset di 1 milione di esempi ordinati per data, `shuffle(100)`. Cosa
   contengono i batch, e perché è un problema?
2. Perché `shuffle` va prima di `batch` e non dopo?
3. Cosa fa `prefetch` e quando NON serve?

<details>
<summary><b>Apri le risposte</b></summary>

1. Esempi quasi consecutivi nel tempo: il buffer da 100 su un milione
   mescola solo localmente. Il modello vede i periodi in sequenza e i
   gradienti sono distorti da bias temporali.
2. Dopo `batch`, shuffle mescolerebbe l'ordine dei *batch interi*, ma ogni
   batch conterrebbe sempre gli stessi esempi vicini tra loro: la
   composizione dei batch non cambierebbe mai tra le epoche.
3. Prepara i batch successivi in parallelo al training. Non serve quando i
   dati sono già tutti in memoria e senza trasformazioni costose — come il
   nostro caso di oggi: qui e' igiene, non necessita'.
</details>

## Fonti

- TensorFlow, *tf.data: Build TensorFlow input pipelines*:
  https://www.tensorflow.org/guide/data
- TensorFlow, *Better performance with the tf.data API*:
  https://www.tensorflow.org/guide/data_performance

## Prossima lezione

La pipeline sa trasformare esempi al volo — e adesso ne abbiamo bisogno
davvero: la **Fase 3** comincia con la tokenizzazione, e il modello legge
finalmente il **testo** delle memorie. Preparati a veder saltare
l'accuratezza.